# Financial Analytics Platform — EDA

Exploratory Data Analysis: загрузка данных, визуализация, проверка качества.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from ingestion.connectors.binance import BinanceConnector
from ingestion.connectors.moex import MoexConnector
from ingestion.validators.ohlcv_validator import OHLCVValidator
from models.anomaly.ensemble import AnomalyEnsemble
from processing.features.feature_calculator import calculate_features

%matplotlib inline

In [ ]:
moex = MoexConnector()
df = moex.fetch_ohlcv(['SBER', 'GAZP'], interval='10m', start_date='2024-06-01', end_date='2024-06-10')
print(f'MOEX: {len(df)} rows')
df.head()

In [ ]:
validator = OHLCVValidator()
result = validator.validate(df)
print(f'Validation passed: {result.passed}')

In [ ]:
features = calculate_features(df)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for symbol in features['symbol'].unique():
    s = features[features['symbol'] == symbol]
    axes[0].plot(s['ts'], s['close'], label=symbol)
    axes[1].plot(s['ts'], s['volatility_14d'], label=symbol)
axes[0].set_title('Close Price')
axes[0].legend()
axes[1].set_title('Volatility 14d')
plt.tight_layout()
plt.show()

In [ ]:
ensemble = AnomalyEnsemble(min_votes=2)
anomalies = ensemble.detect(features)
print(f'Аномалий: {len(anomalies)}')
anomalies.head()